<a href="https://colab.research.google.com/github/Sola-mb/Deeplearning/blob/main/rapport2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow numpy matplotlib scikit-learn seaborn

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, Concatenate
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import json
import io
from google.colab import files

In [ ]:
print("Please upload your tokenizer.json file")
uploaded = files.upload()
tokenizer_json = next(iter(uploaded))
with open(tokenizer_json, 'r', encoding='utf-8') as f:
    tokenizer = tokenizer_from_json(f.read())

print("Please upload X_train_pad.npy, y_train.npy, X_val_pad.npy, y_val.npy, X_test_pad.npy, y_test.npy")
uploaded = files.upload()


X_train = np.load(io.BytesIO(uploaded['X_train_pad.npy']))
y_train = np.load(io.BytesIO(uploaded['y_train_pad.npy']))   # adjust filename if different
X_val = np.load(io.BytesIO(uploaded['X_val_pad.npy']))
y_val = np.load(io.BytesIO(uploaded['y_val_pad.npy']))
X_test = np.load(io.BytesIO(uploaded['X_test_pad.npy']))
y_test = np.load(io.BytesIO(uploaded['y_test_pad.npy']))

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


In [ ]:
MAX_WORDS = 20000
MAX_LEN = 150
EMBEDDING_DIM = 100
BATCH_SIZE = 64
EPOCHS = 10
VOCAB_SIZE = MAX_WORDS + 1

In [ ]:
def build_multikernel_cnn(vocab_size, max_len, embedding_dim=100):
    inputs = Input(shape=(max_len,))
    embedding = Embedding(vocab_size, embedding_dim, input_length=max_len)(inputs)
    conv_3 = Conv1D(128, 3, activation='relu', padding='same')(embedding)
    conv_5 = Conv1D(128, 5, activation='relu', padding='same')(embedding)
    conv_7 = Conv1D(128, 7, activation='relu', padding='same')(embedding)
    concat = Concatenate()([conv_3, conv_5, conv_7])
    pool = GlobalMaxPooling1D()(concat)
    dense = Dense(64, activation='relu')(pool)
    dropout = Dropout(0.5)(dense)
    outputs = Dense(1, activation='sigmoid')(dropout)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

model = build_multikernel_cnn(VOCAB_SIZE, MAX_LEN, EMBEDDING_DIM)
model.summary()


In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
checkpoint = ModelCheckpoint('best_model.keras', monitor='val_accuracy', save_best_only=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop, checkpoint],
    verbose=1
)


In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\n Test Accuracy: {test_acc:.4f}")
print(f" Test Loss: {test_loss:.4f}")

In [ ]:
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = (y_pred_prob >= 0.5).astype(int).flatten()
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))
# Plot confusion matrix
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy Curves')
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss Curves')
plt.show()


In [ ]:
!pip install ipywidgets
from ipywidgets import interact, widgets
import re
from tensorflow.keras.preprocessing.sequence import pad_sequences

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text

def predict_sentiment(review):
    cleaned = clean_text(review)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding='pre', truncating='pre')
    prob = model.predict(padded, verbose=0)[0][0]
    sentiment = "Positive " if prob >= 0.5 else "Negative "
    return sentiment, prob

@interact(review=widgets.Textarea(value="This product is amazing!", description="Review:"))
def show_prediction(review):
    if review.strip():
        sent, conf = predict_sentiment(review)
        print(f"Sentiment: {sent} (confidence: {conf:.4f})")
    else:
        print("Please enter a review.")